<table>
<tr>
    <td width=10%><img src="./img/logo_lri.jpg"></td>
    <td width=30%><img src="./img/medi-chal_logo.png" height=15%></img></td>
</tr>

</table>

<br/><br/><div id="top"></div>

<center><a style="font-size: 40pt; font-weight: bold">To be or not to be?</a></center>
<br/>
<center><a style="font-size: 22pt; font-weight: bold">Binary Classification on Artificial Medical Data</a></center>
<br/>

---

# TP1 Instructions


The goal of this TP is to work on a mortality prediction problem and to participate into the "To be or not to be?" challenge.

A fresh instance of the challenge can be found here: https://competitions.codalab.org/competitions/27605

* **Step 1:** Sign up to https://competitions.codalab.org
* **Step 2:** Go through the competition's pages and read them carefully in order to understand the problem.
* **Step 3:** Download the dataset from "Participate > Files > Public Data" page. Put the "public_data" folder in the same folder as the folder containing this notebook.
* **Step 4:** Go though this notebook. You'll have to read, run, **code your own methods** and **answer quetions**. Use "code" cell for coding and "markdown" cell to answer questions. 


<b><font size="5">Outline</font></b><br/><br/>
&emsp;&emsp;&emsp; I) <a href="#data"> Load data</a><br/>
&emsp;&emsp;&emsp; II) <a href="#exploratory"> Exploratory analysis </a><br/>
&emsp;&emsp;&emsp; III) <a href="#preprocessing"> Preprocessing </a><br/>
&emsp;&emsp;&emsp; IV) <a href="#processing"> Processing </a><br/>
&emsp;&emsp;&emsp; V) <a href="#model"> Predictive model </a><br/>
&emsp;&emsp;&emsp; VI) <a href="#save"> Save predictions </a><br/>

2020 december, 1rst

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import time
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np

import sklearn

# IMPORTS
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    plot_confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from sklearn.svm import LinearSVC

import os
from pathlib import Path

from IPython.display import display
from pprint import pprint

### Question 1: Briefly explain the task.

This is a binary classification of tabular data.  
From information about patients, we need to predict whether the patients will die during their stay in the hospital.  
A target value of 0 means survival, and 1 means death.

### Question 2: What is the metric used in the challenge?

Balanced accuracy: average of accuracy for each class  

<table>
    <tr>
        <td>
            <img src="https://statisticaloddsandends.files.wordpress.com/2020/01/metrics1.png?w=600&h=434" width="250">
        </td>
        <td>
            <img src="https://statisticaloddsandends.files.wordpress.com/2020/01/metrics2.png?w=768&h=352" width="450">
        </td>
    </tr>
</table>
    
$$ 
\texttt{Recall = Sensitivity = True Positive Rate} \\
= \frac{True Positive}{True Positive + False Negative} = \frac{True Positive}{All Actual Positive}
\\
$$

$$
\texttt{Specificity = True Negative Rate = 1- False Positive Rate} \\
= \frac{True Negative}{True Negative + False Positive} = \frac{True Negative}{All Actual Negative}
\\
$$   

$$
\texttt{balanced-accuracy} = \frac 1 2 (\texttt{True Positive Rate} + \texttt{True Negative Rate})
= \frac{1}{2}\left( \frac{TP}{TP + FN} + \frac{TN}{TN + FP}\right )
\\
$$

Reference:  
https://scikit-learn.org/stable/modules/model_evaluation.html#balanced-accuracy-score  
https://mvpa.blogspot.com/2015/12/balanced-accuracy-what-and-why.html  


Example:  
https://statisticaloddsandends.wordpress.com/2020/01/23/what-is-balanced-accuracy/  


In [ ]:
print(os.getcwd())
print(os.listdir("public_data"))

## <a id="data"> I) Load data</a>  

In [ ]:
features = pd.read_csv(
    "public_data/mimic_synthetic_feat.name", header=None
).values.flatten()
labels = pd.read_csv(
    "public_data/mimic_synthetic_label.name", header=None
).values.flatten()
X = pd.read_csv(
    "public_data/mimic_synthetic_train.data",
    header=None,
    names=features,
    sep=" ",
)
y = pd.Series(
    pd.read_csv(
        "public_data/mimic_synthetic_train.solution",
        header=None,
        names=labels,
        sep=" ",
    ).values.flatten()
)

In [ ]:
print(X.shape)
display(X.sample(5))

In [ ]:
print(y.shape)
display(y.sample(5))
display(y.value_counts())

This is the **training set**. This means that have access to the labels and you can use it in anyway you want to prepare your model.

Later on, you'll make your model draws predictions from the **testing set** in order to upload them on Codalab and get a score on unseen labels.

In [ ]:
# Load testing set
X_test = pd.read_csv(
    "public_data/mimic_synthetic_test.data",
    header=None,
    names=features,
    sep=" ",
)

In [ ]:
print(X_test.shape)
# print(X_test.dropna().shape)
display(X_test.sample(5))

## <a id="exploratory"> II) Exploratory analysis</a>

#### Variable types

In [ ]:
types = pd.read_csv(
    "public_data/mimic_synthetic_feat.type", header=None
).set_index(features)[0]
print(types.value_counts())

### Code 1: Display the following plots:
* Bar plots of some categorical variables (e.g. "LANGUAGE", "MARITAL_STATUS")

* Show the correlation between some variables and the target variable y.

* Another plot or any exploratory analysis you can think of

In [ ]:
# TODO: plot1
display(types[types == "Categorical"].index.values)

categorical_features = "ICU INSURANCE GENDER".split(" ")

In [ ]:
X["ICU"].value_counts().plot(kind="bar")

In [ ]:
X["INSURANCE"].value_counts().plot(kind="bar")

In [ ]:
def split_yindex(xdata, ydata):
    """ Split target y into label 0 and label 1"""
    y1_idx = ydata[ydata==1].index
    y0_idx = np.setdiff1d(ydata.index, y1_idx)
    return y0_idx, y1_idx

yindex = split_yindex(X, y)
print(len(yindex[0]), len(yindex[1]))

In [ ]:
def sample_data(xdata, ydata, yindex=None, yratio=4):
    """ Keep all label 1, sample from  label 0 """
    if not yindex: yindex = split_yindex(xdata, ydata)    
    sample_y0_idx = np.random.choice(yindex[0], size=len(yindex[1]) * yratio)    
    sample_idx =  np.concatenate((yindex[1], sample_y0_idx))
    return xdata.iloc[sample_idx], ydata.iloc[sample_idx]

xsample, ysample = sample_data(X, y)
print(xsample.shape, ysample.shape)
display(ysample.value_counts())

In [ ]:
xtrain, xvalid, ytrain, yvalid = train_test_split(xsample, ysample, test_size=0.2)


In [ ]:
def handle_missing(xdata):
    return xdata.fillna(xdata.median())

xtrain_no_missing = handle_missing(xtrain)
xvalid_no_missing = handle_missing(xvalid)
print(xtrain_no_missing.shape, xvalid_no_missing.shape)

In [ ]:
"""
def encode_categorical(xdata):
#     cat_code_cols = []
    cat_cols = "ICU INSURANCE GENDER".split(" ")
    xencd = xdata.copy()
    for cat_col in cat_cols:
        cat_code = cat_col + "_code"
        xencd[cat_code] = xencd[cat_col].astype("category").cat.codes
#         cat_code_cols.append(cat_code)
    xencd = xencd.drop(columns=cat_cols)
    return xencd

# xtrain_encoded, cat_cols = encode_categorical(xtrain_no_missing, categorical_features)
# xvalid_encoded, _ = encode_categorical(xvalid_no_missing, categorical_features)
xtrain_encoded = encode_categorical(xtrain_no_missing)
xvalid_encoded = encode_categorical(xvalid_no_missing)
print(xtrain_encoded.shape)
display(xtrain_encoded.sample(5)["ICU_code"])
"""

In [ ]:
cat_sample = X.sample(5)
cat_onehot = pd.get_dummies(cat_sample, columns=["ICU", "INSURANCE"], dummy_na=True)
display(cat_onehot)
print(cat_sample.shape, cat_onehot.shape)

## <a id="preprocessing"> III) Preprocessing </a>

In [ ]:
def encode_categorical(xdata):
    xencoded = xdata.copy()
    xencoded["is_male"] = (xdata["GENDER"] == "M").astype(int)
    
    onehot_cols = "ICU INSURANCE".split(" ")
    xencoded = pd.get_dummies(xencoded, columns=onehot_cols, dummy_na=True)
    
    return xencoded

xvalid_encoded = encode_categorical(xvalid_no_missing)
print(xvalid_no_missing.shape, xvalid_encoded.shape)
print(xvalid_encoded.sample(5)["is_male"])

In [ ]:
# types[types == "Numerical"][:20]
numerical_features = types[types == "Numerical"].index.values
print(numerical_features[0:10])
print(type(numerical_features))

In [ ]:
binary_features = types[types == "Binary"].index.values
binary_features = binary_features[binary_features != "GENDER"]
# bin_mutual_info = mutual_info_classif(Xcleaned.iloc[sample_idx][binary_features], ycleaned[sample_idx])

In [ ]:

# selected_columns = np.concatenate((numerical_features, binary_features, cat_cols))
# print(selected_columns.shape)

In [ ]:
def select_columns(xdata):
    exclude_cols = ('ADMITTIME', 'DISCHTIME', 'LANGUAGE', 'RELIGION', 'MARITAL_STATUS', 'ETHNICITY', 'GENDER')
    selected_cols = np.setdiff1d(xdata.columns, exclude_cols)
#     print(len(xdata.columns), len(selected_cols))
    return xdata[selected_cols]

print(select_columns(xvalid_encoded).shape)    

In [ ]:
def preprocess(xraw):
    xencoded = encode_categorical(xraw)
    x_no_missing = handle_missing(xencoded)
    xselected = select_columns(x_no_missing)
    return xselected

xtrain_preprocess = preprocess(xtrain)
xvalid_preprocess = preprocess(xvalid)
print(xtrain_preprocess.shape, xvalid_preprocess.shape)


In [ ]:
pca_model = sklearn.decomposition.PCA()
xtrain_pca = pca_model.fit_transform(xtrain_preprocess)

In [ ]:
print(pca_model.explained_variance_ratio_[:30])

In [ ]:
plt.scatter(xtrain_pca[:,0], xtrain_pca[:,1], c=ytrain)
plt.colorbar()
plt.show()

## <a id="processing"> III) Processing</a>

As you may have noticed, various variables are categorical and encoded with strings. Most models won't be able to take them as input.

### Code 2: Transform the categorical data into integers using the encoding of your choice.

_Hint: The simplest methods is called "Label Encoding". It simply consists in replacing each new category you encounter by the integer, starting from 0. It may not be the most efficient method._

In [ ]:
# xtrain_final = xtrain_encoded[selected_columns]
xtrain_final = xtrain_preprocess
ytrain_final = ytrain

# xvalid_final = xvalid_encoded[selected_columns]
xvalid_final = xvalid_preprocess
yvalid_final = yvalid

print(xtrain_final.shape, ytrain_final.shape)
print(xvalid_final.shape, yvalid_final.shape)

## <a id="model"> IV) Predictive model</a>
Train a model and assess its performance.

### Random Forest Classifier model

In [ ]:
# Baseline: Random Forest
clf = RandomForestClassifier(random_state = 42,
    n_estimators=10, class_weight="balanced",
    max_depth=5, 
#     min_samples_split=10,
#     min_samples_leaf=10,
#     max_leaf_nodes=20,
)


clf.fit(xtrain_final, ytrain_final)  # training

ypred = clf.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

In [ ]:
plot_confusion_matrix(clf, xvalid_final, yvalid_final)

In [ ]:
n_components = np.arange(start=10, stop=300, step=10)
pca_scores = []

for n_component in n_components:
    pca_clf = RandomForestClassifier(random_state=42, 
        n_estimators=10, class_weight="balanced",
        max_depth=5, 
    )

    pca_clf.fit(xtrain_pca[:, :n_component], ytrain_final)  # training

    pca_predictions = pca_clf.predict(pca_model.transform(xvalid_final)[:, :n_component])  # predictions
    pca_scores.append(balanced_accuracy_score(yvalid_final, pca_predictions))  # scoring
    


In [ ]:
plt.plot(n_components, pca_scores)
plt.xlabel("PCA # components")
plt.ylabel("valid balanced accuracy")

In [ ]:
#the list of index ordered by values of pca_scores asc
sorted(range(len(pca_scores)), key=lambda k: pca_scores[k])

In [ ]:
print('n_components: ', n_components[4], ' and pca_scores: ', pca_scores[4])

### Code 3: the code above uses a simple "hold out" method to get the validation score. Use cross-validation to get a more stable estimation.

In [ ]:
# TODO: cross-validation

In [ ]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(clf, xtrain_final, ytrain_final, cv=10, scoring='balanced_accuracy')
sorted(cv_results.keys())

In [ ]:
print(cv_results['test_score'])
print("The average score of cross Validation: ", cv_results['test_score'].mean())

### Code 4: Try to improve your score by using at least one of the following methods:

* Hyperparameter search
* Model selection
* Data augmentation

In [ ]:
n_estimators = [int(x) for x in range(100,1500,200)]
test_results = []
for n in n_estimators:
    rforest = RandomForestClassifier(random_state = 42, n_estimators = n)
    rforest.fit(xtrain_final, ytrain_final)
    ypred = rforest.predict(xvalid_final)
    accuracy = balanced_accuracy_score(yvalid_final, ypred)
    test_results.append(accuracy)
from matplotlib.legend_handler import HandlerLine2D
line1, = plt.plot(n_estimators, test_results, 'r', label="Test balanced_accuracy")
plt.legend(handler_map={line1: HandlerLine2D(numpoints=2)})
plt.ylabel('balanced_accuracy_score')
plt.xlabel('RF n_estimators')
plt.show()

In [ ]:
#the list of index ordered by values of test_results asc
print(sorted(range(len(test_results)), key=lambda k: test_results[k]))
print(n_estimators[0])

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Number of trees in random forest
n_estimators = [int(x) for x in range(100,1500,200)]
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 8)]
max_depth.append(None)
# number of features at every split
max_features = ['auto', 'sqrt', 'log2']
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
class_weight = [None, 'balanced', 'balanced_subsample']
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'class_weight': class_weight
              }
# Use the random grid to search for best hyperparameters
# First create the base model to tune
rf = RandomForestClassifier(random_state=42,)
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 10,
                               scoring='balanced_accuracy', cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(xtrain_final, ytrain_final)

In [ ]:
rf_random.best_params_

In [ ]:
rf_clf = RandomForestClassifier(random_state = 42,
    n_estimators = 100, min_samples_split = 2, class_weight= 'balanced_subsample',
    min_samples_leaf = 4, max_features = 'auto', max_depth = 10)


rf_clf.fit(xtrain_final, ytrain_final)  # training

ypred = rf_clf.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

In [ ]:
max_depth = [int(x) for x in np.linspace(10, 110, num = 8)]
max_depth.append(None)
test2_results = []
for m in max_depth:
    rforest = RandomForestClassifier(random_state = 42, n_estimators = 10, max_depth = m)
    rforest.fit(xtrain_final, ytrain_final)
    ypred = rforest.predict(xvalid_final)
    accuracy = balanced_accuracy_score(yvalid_final, ypred)
    test2_results.append(accuracy)
from matplotlib.legend_handler import HandlerLine2D
line2, = plt.plot(max_depth, test2_results, 'r', label="Test balanced_accuracy")
plt.legend(handler_map={line1: HandlerLine2D(numpoints=2)})
plt.ylabel('balanced_accuracy_score')
plt.xlabel('RF max_depth')
plt.show()

In [ ]:
#the list of index ordered by values of test_results asc
print(sorted(range(len(test2_results)), key=lambda k: test2_results[k]))
test2_results[1]

In [ ]:
max_features = ['auto', 'sqrt', 'log2']
test3_results = []
for mf in max_features:
    rforest = RandomForestClassifier(random_state = 42, n_estimators = 100, max_depth = 24, max_features=mf)
    rforest.fit(xtrain_final, ytrain_final)
    ypred = rforest.predict(xvalid_final)
    accuracy = balanced_accuracy_score(yvalid_final, ypred)
    test3_results.append(accuracy)
from matplotlib.legend_handler import HandlerLine2D
line3, = plt.plot(max_features, test3_results, 'r', label="Test balanced_accuracy")
plt.legend(handler_map={line1: HandlerLine2D(numpoints=2)})
plt.ylabel('balanced_accuracy_score')
plt.xlabel('RF max_depth')
plt.show()

In [ ]:
class_weight = [None, 'balanced', 'balanced_subsample']
class_weight2 = ['None', 'balanced', 'balanced_subsample']
test4_results = []
for cw in class_weight:
    rforest = RandomForestClassifier(random_state = 42, n_estimators = 100, max_depth = 24,
                                     max_features='auto', class_weight=cw)
    rforest.fit(xtrain_final, ytrain_final)
    ypred = rforest.predict(xvalid_final)
    accuracy = balanced_accuracy_score(yvalid_final, ypred)
    test4_results.append(accuracy)
from matplotlib.legend_handler import HandlerLine2D
line3, = plt.plot(class_weight2, test4_results, 'r', label="Test balanced_accuracy")
plt.legend(handler_map={line1: HandlerLine2D(numpoints=2)})
plt.ylabel('balanced_accuracy_score')
plt.xlabel('RF class_weight')
plt.show()

In [ ]:
test4_results[0]

In [ ]:
_clf = RandomForestClassifier(random_state = 42,
    n_estimators = 10, min_samples_split = 10, class_weight="balanced",
    min_samples_leaf = 2, max_features = 'sqrt', max_depth = 5)


_clf.fit(xtrain_final, ytrain_final)  # training

_ypred = _clf.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, _ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

In [ ]:
"""from sklearn.model_selection import RandomizedSearchCV

# Number of trees in random forest
n_estimators = [int(x) for x in range(200,2000,200)]
n_estimators.extend([int(x) for x in range(10,2000,250)])
sorted(n_estimators)

# Number of features to consider at every split
max_features = ['auto', 'sqrt', 'log2']

# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None)

# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]

# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]

# Method of selecting samples for training each tree
bootstrap = [True, False]

#
class_weight = [None, 'balanced', 'balanced_subsample']

# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap,
               'class_weight': class_weight}

# Use the random grid to search for best hyperparameters
# First create the base model to tune
rf = RandomForestClassifier(random_state=42,)
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(xtrain_final, ytrain_final)

rf_random.best_estimator_
rf_random.best_params_
"""

In [ ]:
def evaluate(model, x_test, y_test):
    y_pred = model.predict(x_test)
    accuracy = balanced_accuracy_score(y_test, y_pred)
    print (accuracy)
    print(confusion_matrix(y_test,y_pred))
    

#best_random = rf_random.best_estimator_
#evaluate(best_random, xvalid_final, yvalid_final)

In [ ]:
from sklearn.model_selection import validation_curve
n_estimators = [100, 300, 500, 750, 800, 1200]
train_scores, test_scores = validation_curve(
                                RandomForestClassifier(),
                                X = xtrain_final, y = ytrain_final, 
                                param_name = 'n_estimators', 
                                param_range = n_estimators, cv = 3,scoring="balanced_accuracy", n_jobs=1)

In [ ]:
#the list of index ordered by values of test_results asc
#print(sorted(range(len(test_score)), key=lambda k: test_score[k]))
print(test_scores)

In [ ]:
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.title("Validation Curve with RFC")
plt.xlabel("n_estimators")
plt.ylabel("Score")
plt.ylim(0.0, 1.1)
plt.semilogx(n_estimators, train_scores, label="Training score", color="r")
plt.fill_between(n_estimators, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.2, color="r")
plt.semilogx(n_estimators, test_scores_mean, label="Cross-validation score",
             color="g")
plt.fill_between(n_estimators, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.2, color="g")
plt.legend(loc="best")
plt.show()

### Question 3: What is up-sampling? Why it may be worth a try for this problem?

TODO: answer

In [ ]:
# check version number
! pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

In [ ]:
#up-sampling training data 
os = SMOTE(random_state=0)
columns = xtrain_final.columns
os_data_X, os_data_y = os.fit_sample(xtrain_final, ytrain_final)
os_data_X = pd.DataFrame(data=os_data_X,columns=columns )
os_data_y= pd.DataFrame(data=os_data_y,columns=['y'])
# we can Check the numbers of our data
print("length of oversampled data is ",len(os_data_X))
print("Number of no subscription in oversampled data",len(os_data_y[os_data_y['y']==0]))
print("Number of subscription",len(os_data_y[os_data_y['y']==1]))
print("Proportion of no subscription data in oversampled data is ",len(os_data_y[os_data_y['y']==0])/len(os_data_X))
print("Proportion of subscription data in oversampled data is ",len(os_data_y[os_data_y['y']==1])/len(os_data_X))

In [ ]:
rf_random_clf = RandomForestClassifier(random_state = 42,
    n_estimators = 100, min_samples_split = 2, class_weight= 'balanced_subsample',
    min_samples_leaf = 4, max_features = 'auto', max_depth = 10)


rf_random_clf.fit(os_data_X, os_data_y.values.ravel())  # training

ypred = rf_random_clf.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

In [ ]:
# Baseline: Random Forest
clf = RandomForestClassifier(random_state = 42,
    n_estimators=10, class_weight="balanced",
    max_depth=5, 
#     min_samples_split=10,
#     min_samples_leaf=10,
#     max_leaf_nodes=20,
)


clf.fit(os_data_X, os_data_y.values.ravel())  # training

ypred = clf.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

In [ ]:
#tuning hyperparameters for up-sampled training data

from sklearn.model_selection import RandomizedSearchCV

# Number of trees in random forest
n_estimators = [int(x) for x in range(100,1500,200)]
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 8)]
max_depth.append(None)
# number of features at every split
max_features = ['auto', 'sqrt', 'log2']
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
class_weight = [None, 'balanced', 'balanced_subsample']
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'class_weight': class_weight
              }
# Use the random grid to search for best hyperparameters
# First create the base model to tune
uprf = RandomForestClassifier(random_state=42,)
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
uprf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 10,
                               scoring='balanced_accuracy', cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
uprf_random.fit(os_data_X, os_data_y.values.ravel())

In [ ]:
uprf_random.best_params_

In [ ]:
# Best random forest classifier after up-sampling
up_clf = RandomForestClassifier(random_state = 42,
    n_estimators=1100, class_weight="balanced",
    min_samples_split = 2, min_samples_leaf = 1,
    max_depth=110, max_features = 'sqrt')


up_clf.fit(os_data_X, os_data_y.values.ravel())  # training

ypred = up_clf.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

### Decision Tree Classifier model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(random_state= 42)
tree.fit(xtrain_final, ytrain_final)

ypred = tree.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(xtrain_final, ytrain_final)
y_pred = dt.predict(xvalid_final)

max_depths = np.linspace(1, 32, 32, endpoint=True)
train_results = []
test_results = []
for max_depth in max_depths:
   dt = DecisionTreeClassifier(random_state = 42, max_depth=max_depth)
   dt.fit(xtrain_final, ytrain_final)
   train_pred = dt.predict(xtrain_final)
   accuracy = balanced_accuracy_score(ytrain_final, train_pred)
   train_results.append(accuracy)
   y_pred = dt.predict(xvalid_final)
   accuracy = balanced_accuracy_score(yvalid_final, y_pred)
   # Add auc score to previous test results
   test_results.append(accuracy)

from matplotlib.legend_handler import HandlerLine2D
line1, = plt.plot(max_depths, train_results, 'b', label="Train Accuracy")
line2, = plt.plot(max_depths, test_results, 'r', label="Test Accuracy")
plt.legend(handler_map={line1: HandlerLine2D(numpoints=2)})
plt.ylabel('Accuracy score')
plt.xlabel('Tree depth')
plt.show()

In [ ]:
min_samples_splits = np.linspace(0.1, 1.0, 10, endpoint=True)
train_results = []
test_results = []
for min_samples_split in min_samples_splits:
   dt = DecisionTreeClassifier(random_state = 42, min_samples_split=min_samples_split)
   dt.fit(xtrain_final, ytrain_final)
   train_pred = dt.predict(xtrain_final)
   accuracy = balanced_accuracy_score(ytrain_final, train_pred)
   train_results.append(accuracy)
   y_pred = dt.predict(xvalid_final)
   accuracy = balanced_accuracy_score(yvalid_final, y_pred)
   # Add auc score to previous test results
   test_results.append(accuracy)

from matplotlib.legend_handler import HandlerLine2D
line1, = plt.plot(min_samples_splits, train_results, 'b', label="Train Accuracy")
line2, = plt.plot(min_samples_splits, test_results, 'r', label="Test Accuracy")
plt.legend(handler_map={line1: HandlerLine2D(numpoints=2)})
plt.ylabel('Accuracy score')
plt.xlabel('Tree min_samples_split')
plt.show()

In [ ]:
tree_ = DecisionTreeClassifier(random_state = 42, max_depth=1000, max_features=250)


tree_.fit(xtrain_final, ytrain_final)  # training

ypred = tree_.predict(xvalid_final)  # predictions
score = balanced_accuracy_score(yvalid_final, ypred)  # scoring
print("Balanced accuracy score: {}".format(score))

## <a id="save"> V) Save Predictions</a>

* Use the following code to make predictions and save them as a ZIP file.
* Try to upload this ZIP file into the competition ("Participate > Submit/ View Results > Submit")
* Do your best to be **on top of the leaderboard!**

In [ ]:
xtest_preprocess = preprocess(X_test)
predictions = rf_clf.predict(xtest_preprocess)  # compute predictions (0 and 1)


In [ ]:
predictions_file = "mimic_synthetic_test.csv"
pd.Series(predictions).to_csv(predictions_file, index=False, header=False)
print("Predictions saved.")

t = time.asctime().replace(" ", "_").replace(":", "-")
output_file = "./submissions/submission_" + t + ".zip"

# !zip test_submission.zip mimic_synthetic_test.csv # create a ZIP

with ZipFile(output_file, "w") as z:
    z.write(predictions_file)
print("The submission is ready: {}".format(output_file))

### Interesting to know!

The dataset doesn't contain actual medical data. We are not allowed to share with you such **confidential data** from critical care patients. To avoid that problem, we replaced the data by **artificial data**. 

In order to have credible data, we have generated them using a **Wasserstein GAN**.

For more information, here is a reference: <br />
Privacy Preserving Synthetic Health Data <br />
Andrew Yale, Saloni Dash, Ritik Dutta, Isabelle Guyon, Adrien Pavao, Kristin Bennett <br />
https://hal.inria.fr/hal-02160496/document